### Imports

In [1]:
import os
import json
import random
import shutil
import tempfile

import torch

from huggingface_hub import login, HfApi, hf_hub_download
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
!pip install -q tokenizers 
!pip install ijson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.8/149.8 kB 3.4 MB/s eta 0:00:00


In [3]:
!pip install -q bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.7 MB/s eta 0:00:00


In [4]:
import re
import ijson
from collections import Counter

from tokenizers import ByteLevelBPETokenizer

## Configuration


In [5]:
# =========================
# PROJECT CONFIGURATION
# =========================

# Hugging Face
HF_USERNAME = "A-240031"
HF_REPO_NAME = "legal-lm-phase1"
HF_MODEL_REPO_ID = f"{HF_USERNAME}/{HF_REPO_NAME}"
HF_DATASET_REPO_ID = "Exploration-Lab/CS-781-Capstone"
# Dataset files
DATASET_FILES = {
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "high_courts": "Phase_1_Pretraining/Train/train_high_courts.json",
    "acts": "Phase_1_Pretraining/Train/train_acts.json",
    "tribunals": "Phase_1_Pretraining/Train/train_tribunals.json",
}

# Checkpointing
CHECKPOINT_INTERVAL_TOKENS = 1_000_000

# Device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


#TOKENIZER CONFIGURATION
VOCAB_SIZE = 32_768
MIN_FREQUENCY = 5
CUSTOM_TOKEN_MIN_FREQUENCY = 100

SPECIAL_TOKENS = [
    "<|pad|>",
    "<|unk|>",
    "<|bos|>",
    "<|eos|>",
]

ARCH_TRAIN_TOKENS = 50_000_000
VAL_EXAMPLES = 500
VAL_PROMPT_WORDS = 512
VAL_TARGET_WORDS = 128
VAL_DOC_FRACTION = 0.01
SEED = 42

ARCH_TRAIN_PATH = "/kaggle/working/architecture_data/arch_train.bin"
ARCH_VAL_PATH = "/kaggle/working/architecture_data/arch_val.json"
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

# =========================
# TRAINING CONFIGURATION
# =========================


In [6]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [7]:
BUCKET_NAME = "A-240031/legal-lm-phase1-bucket"
BUCKET_ROOT = f"hf://buckets/{BUCKET_NAME}"

from huggingface_hub import HfFileSystem

fs = HfFileSystem()

In [8]:
def upload_to_bucket(local_path, remote_path):
    remote_path = remote_path.lstrip("/")

    fs.put_file(
        local_path,
        f"{BUCKET_ROOT}/{remote_path}",
    )

    print(f"Uploaded: {remote_path}")

In [9]:
def download_from_bucket(remote_path, local_path):
    remote_path = remote_path.lstrip("/")

    fs.get_file(
        f"{BUCKET_ROOT}/{remote_path}",
        local_path,
    )

    print(f"Downloaded: {remote_path} -> {local_path}")

## Checkpoint related functions

In [10]:
def upload_checkpoint(tmp_dir, tag):

    remote_dir = f"checkpoints/{tag}"

    for filename in os.listdir(tmp_dir):
        local_path = os.path.join(tmp_dir, filename)

        upload_to_bucket(
            local_path,
            f"{remote_dir}/{filename}",
        )

In [11]:
def save_checkpoint(tag, model, optimizer, scheduler=None, scaler=None,
                    tokens_seen=0, global_step=0, epoch=0, extra_state=None):

    with tempfile.TemporaryDirectory() as tmp_dir:

        torch.save(model.state_dict(), os.path.join(tmp_dir, "model.pt"))
        torch.save(optimizer.state_dict(), os.path.join(tmp_dir, "optimizer.pt"))

        if scheduler is not None:
            torch.save(scheduler.state_dict(), os.path.join(tmp_dir, "scheduler.pt"))

        if scaler is not None:
            torch.save(scaler.state_dict(), os.path.join(tmp_dir, "scaler.pt"))

        rng_state = {"python": random.getstate(), "torch": torch.get_rng_state()}

        if torch.cuda.is_available():
            rng_state["cuda"] = torch.cuda.get_rng_state_all()

        torch.save(rng_state, os.path.join(tmp_dir, "rng_state.pt"))

        state = {
            "tag": tag,
            "tokens_seen": tokens_seen,
            "global_step": global_step,
            "epoch": epoch,
            "extra_state": extra_state,
        }

        with open(os.path.join(tmp_dir, "training_state.json"), "w") as f:
            json.dump(state, f, indent=2)

        upload_checkpoint(tmp_dir,tag)

def load_checkpoint(tag, model, optimizer, scheduler=None, scaler=None):

    with tempfile.TemporaryDirectory() as tmp_dir:

        files = [
            "model.pt",
            "optimizer.pt",
            "scheduler.pt",
            "scaler.pt",
            "rng_state.pt",
            "training_state.json",
        ]

        checkpoint_dir = os.path.join(
            tmp_dir,
            "checkpoints",
            tag,
        )

        for filename in files:
            try:
                download_from_bucket(
                    f"checkpoints/{tag}/{filename}",
                    os.path.join(checkpoint_dir, filename),
                )
            except Exception:
                pass

        model.load_state_dict(
            torch.load(
                os.path.join(checkpoint_dir, "model.pt"),
                map_location=DEVICE,
                weights_only=True,
            )
        )

        optimizer.load_state_dict(
            torch.load(
                os.path.join(checkpoint_dir, "optimizer.pt"),
                map_location=DEVICE,
                weights_only=False,
            )
        )

        scheduler_path = os.path.join(checkpoint_dir, "scheduler.pt")

        if scheduler is not None and os.path.exists(scheduler_path):
            scheduler.load_state_dict(
                torch.load(
                    scheduler_path,
                    map_location=DEVICE,
                    weights_only=False,
                )
            )

        scaler_path = os.path.join(checkpoint_dir, "scaler.pt")

        if scaler is not None and os.path.exists(scaler_path):
            scaler.load_state_dict(
                torch.load(
                    scaler_path,
                    map_location=DEVICE,
                    weights_only=False,
                )
            )

        rng_state = torch.load(
            os.path.join(checkpoint_dir, "rng_state.pt"),
            map_location="cpu",
            weights_only=False,
        )

        random.setstate(rng_state["python"])
        torch.set_rng_state(rng_state["torch"])

        if torch.cuda.is_available() and "cuda" in rng_state:
            torch.cuda.set_rng_state_all(rng_state["cuda"])

        with open(
            os.path.join(checkpoint_dir, "training_state.json")
        ) as f:
            state = json.load(f)

    return state

## Loading training data

In [12]:
MIXED_DATASET_PATH = "/kaggle/working/mixed_100m.jsonl"

download_from_bucket(
    "architecture_data/mixed_100m.jsonl",
    MIXED_DATASET_PATH,
)

(…)ucket/architecture_data/mixed_100m.jsonl:   0%|          | 0.00/597M [00:00<?, ?B/s]

Downloaded: architecture_data/mixed_100m.jsonl -> /kaggle/working/mixed_100m.jsonl


In [13]:
def corpus_iterator(path):
    import json

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            text = record.get("text")
            if text:
                yield text


In [14]:
tokenizer = ByteLevelBPETokenizer(add_prefix_space=False,lowercase=False)
tokenizer.train_from_iterator(corpus_iterator(MIXED_DATASET_PATH),vocab_size=VOCAB_SIZE,min_frequency=MIN_FREQUENCY,special_tokens=SPECIAL_TOKENS)

In [15]:
tokenizer

Tokenizer(vocabulary_size=32768, model=ByteLevelBPE, add_prefix_space=False, lowercase=False, dropout=None, unicode_normalizer=None, continuing_subword_prefix=None, end_of_word_suffix=None, trim_offsets=False)

In [16]:
def add_custom_legal_tokens(tokenizer, corpus_path, min_frequency=100):
    patterns = [
        re.compile(r"\b(?:Article|ARTICLE)\s+\d+[A-Za-z]*(?:\([0-9A-Za-z]+\))*"),
        re.compile(r"\b(?:Section|SECTION)\s+\d+[A-Za-z]*(?:\([0-9A-Za-z]+\))*"),
        re.compile(r"\b(?:Rule|RULE)\s+\d+[A-Za-z]*(?:\([0-9A-Za-z]+\))*"),
        re.compile(r"\b(?:Regulation|REGULATION)\s+\d+[A-Za-z]*(?:\([0-9A-Za-z]+\))*"),
        re.compile(r"\([a-z]\)"),
        re.compile(r"\([0-9]{1,2}\)"),
        re.compile(r"\((?:i{1,3}|iv|v|vi{0,3}|ix|x{1,2})\)"),
    ]

    counts = Counter()

    for text in corpus_iterator(corpus_path):
        for pattern in patterns:
            counts.update(pattern.findall(text))

    custom_tokens = []

    for token, count in counts.items():
        token_with_space = "Ġ" + token

        if count >= min_frequency and tokenizer.token_to_id(token_with_space) is None:
            custom_tokens.append(token_with_space)

    tokenizer.add_tokens(custom_tokens)

    return custom_tokens, counts

In [17]:
custom_tokens, custom_counts = add_custom_legal_tokens(tokenizer,MIXED_DATASET_PATH,CUSTOM_TOKEN_MIN_FREQUENCY)

print("Custom tokens added:", len(custom_tokens))
print("Final vocabulary size:", tokenizer.get_vocab_size())

Custom tokens added: 410
Final vocabulary size: 33178


In [18]:
VOCAB_SIZE = tokenizer.get_vocab_size()

In [19]:
from pathlib import Path
from huggingface_hub import HfApi

TOKENIZER_DIR = Path("tokenizer")
TOKENIZER_DIR.mkdir(exist_ok=True)

tokenizer.save_model(str(TOKENIZER_DIR))
tokenizer.save(str(TOKENIZER_DIR / "tokenizer.json"))

# HfApi().upload_folder(
#     folder_path=str(TOKENIZER_DIR),
#     repo_id=HF_MODEL_REPO_ID,
#     repo_type="model",
#     path_in_repo="tokenizer",
# )
upload_to_bucket(str(TOKENIZER_DIR),"tokenizer")
print("Tokenizer saved to:", f"{HF_MODEL_REPO_ID}/tokenizer")

Uploaded: tokenizer
Tokenizer saved to: A-240031/legal-lm-phase1/tokenizer


## Splitting dataset into train and val set for choosing architecture

In [20]:
def load_or_create_architecture_data(mixed_path):

    try:
        download_from_bucket(
                "architecture_data/arch_train_mixed.bin",
                ARCH_TRAIN_PATH,
            )
            
        download_from_bucket(
                "architecture_data/arch_val_mixed.json",
                ARCH_VAL_PATH,
            )

        print("Mixed architecture data loaded from Hugging Face.")
        return

    except Exception:
        print("Mixed architecture data not found. Creating from mixed_100m...")

    

    # Count documents for document-wise split
    num_docs = 0

    with open(mixed_path, "r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)

            if record.get("text"):
                num_docs += 1

    split = int(num_docs * VAL_DOC_FRACTION)

    print(f"Total documents: {num_docs:,}")
    print(f"Validation documents: {split:,}")
    print(f"Training documents: {num_docs - split:,}")

    val_examples = []

    with open(mixed_path, "r", encoding="utf-8") as src, \
         open(ARCH_TRAIN_PATH, "wb") as train_file:

        doc_idx = 0

        for line in src:

            record = json.loads(line)
            text = record.get("text")

            if not text:
                continue

            # -------------------------
            # Validation documents
            # -------------------------
            if doc_idx < split:

                words = text.split()

                if len(words) >= VAL_PROMPT_WORDS + VAL_TARGET_WORDS:

                    for start in range(
                        0,
                        len(words)
                        - VAL_PROMPT_WORDS
                        - VAL_TARGET_WORDS
                        + 1,
                        VAL_PROMPT_WORDS,
                    ):

                        val_examples.append({
                            "prompt": " ".join(
                                words[
                                    start:start + VAL_PROMPT_WORDS
                                ]
                            ),
                            "target": " ".join(
                                words[
                                    start + VAL_PROMPT_WORDS:
                                    start + VAL_PROMPT_WORDS + VAL_TARGET_WORDS
                                ]
                            ),
                        })

                        if len(val_examples) >= VAL_EXAMPLES:
                            break

            # -------------------------
            # Training documents
            # -------------------------
            else:

                ids = tokenizer.encode(text).ids

                ids.append(
                    tokenizer.token_to_id("<|eos|>")
                )

                np.asarray(
                    ids,
                    dtype=np.uint16
                ).tofile(train_file)

            doc_idx += 1

    # Save validation examples
    with open(
        ARCH_VAL_PATH,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            val_examples,
            f,
            ensure_ascii=False
        )

    # Upload mixed architecture training data
    upload_to_bucket(ARCH_TRAIN_PATH,"architecture_data/arch_train_mixed.bin" )

    # Upload mixed architecture validation data
    upload_to_bucket(ARCH_VAL_PATH,"architecture_data/arch_val_mixed.json" )
    print("Mixed architecture data created and uploaded.")

In [21]:
load_or_create_architecture_data(MIXED_DATASET_PATH)


(…)t/architecture_data/arch_train_mixed.bin:   0%|          | 0.00/249M [00:00<?, ?B/s]

Downloaded: architecture_data/arch_train_mixed.bin -> /kaggle/working/architecture_data/arch_train.bin


(…)et/architecture_data/arch_val_mixed.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Downloaded: architecture_data/arch_val_mixed.json -> /kaggle/working/architecture_data/arch_val.json
Mixed architecture data loaded from Hugging Face.


In [22]:
VOCAB_SIZE = tokenizer.get_vocab_size()

D_MODEL = 768
N_LAYERS = 12
N_HEADS = 12
N_KV_HEADS = 4          # GQA: 4 KV heads, 12 query heads
D_FF = 3072
SEQ_LEN = 2048

BATCH_SIZE = 4

LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
GRAD_CLIP = 5.0

ARCH_TRAIN_TOKENS = 100_000_000
CHECKPOINT_INTERVAL_TOKENS = 1_000_000

In [23]:
# =========================
# SEQUENTIAL TOKEN DATASET
# =========================
SEQ_LEN = 2048
import numpy as np
from torch.utils.data import Dataset, DataLoader
class TokenDataset(Dataset):
    def __init__(self, path, seq_len):
        self.tokens = np.memmap(path, dtype=np.uint16, mode="r")
        self.seq_len = seq_len
        self.num_sequences = len(self.tokens) // seq_len

    def __len__(self):
        return self.num_sequences

    def __getitem__(self, idx):
        start = idx * self.seq_len
        end = start + self.seq_len

        return torch.from_numpy(
            self.tokens[start:end].astype(np.int64)
        )


train_dataset = TokenDataset(
    ARCH_TRAIN_PATH,
    SEQ_LEN,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=True,
)

TOKENS_PER_STEP = BATCH_SIZE * SEQ_LEN
TOKENS_PER_EPOCH = CHECKPOINT_INTERVAL_TOKENS
TOTAL_TOKENS = min(
    ARCH_TRAIN_TOKENS,
    len(train_dataset) * SEQ_LEN,
)



In [24]:
with open(ARCH_VAL_PATH, "r", encoding="utf-8") as f:
    val_data = json.load(f)

## Model - A3 

In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# RoPE
# ============================================================

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len=2048, base=10000):
        super().__init__()

        inv_freq = 1.0 / (
            base ** (
                torch.arange(0, head_dim, 2).float() / head_dim
            )
        )

        positions = torch.arange(max_seq_len).float()
        freqs = torch.outer(positions, inv_freq)

        self.register_buffer(
            "cos",
            freqs.cos(),
            persistent=False,
        )

        self.register_buffer(
            "sin",
            freqs.sin(),
            persistent=False,
        )

    def forward(self, q, k, offset=0):

        # q, k: [B, H, T, D]

        T = q.size(2)

        cos = self.cos[offset:offset + T]
        sin = self.sin[offset:offset + T]

        cos = cos.unsqueeze(0).unsqueeze(0)
        sin = sin.unsqueeze(0).unsqueeze(0)

        q1, q2 = q[..., ::2], q[..., 1::2]
        k1, k2 = k[..., ::2], k[..., 1::2]

        q_rot = torch.stack(
            [
                q1 * cos - q2 * sin,
                q1 * sin + q2 * cos,
            ],
            dim=-1,
        ).flatten(-2)

        k_rot = torch.stack(
            [
                k1 * cos - k2 * sin,
                k1 * sin + k2 * cos,
            ],
            dim=-1,
        ).flatten(-2)

        return q_rot, k_rot

# ============================================================
# SwiGLU
# ============================================================

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()

        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.down_proj(
            F.silu(self.gate_proj(x)) * self.up_proj(x)
        )

class MLAAttention(nn.Module):

    def __init__(
        self,
        d_model,
        n_heads,
        kv_latent_dim,
        rope_dim,
        max_seq_len=2048,
    ):
        super().__init__()

        assert d_model % n_heads == 0
        assert rope_dim % 2 == 0
        assert rope_dim < d_model // n_heads

        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.content_dim = self.head_dim - rope_dim
        self.rope_dim = rope_dim
        self.kv_latent_dim = kv_latent_dim

        # --------------------------------------------------
        # Query
        # --------------------------------------------------

        self.q_content_proj = nn.Linear(
            d_model,
            n_heads * self.content_dim,
            bias=False,
        )

        self.q_rope_proj = nn.Linear(
            d_model,
            n_heads * rope_dim,
            bias=False,
        )

        # --------------------------------------------------
        # Compressed KV representation
        # --------------------------------------------------

        self.kv_down_proj = nn.Linear(
            d_model,
            kv_latent_dim,
            bias=False,
        )

        self.k_content_up_proj = nn.Linear(
            kv_latent_dim,
            n_heads * self.content_dim,
            bias=False,
        )

        self.v_up_proj = nn.Linear(
            kv_latent_dim,
            n_heads * self.head_dim,
            bias=False,
        )

        # --------------------------------------------------
        # Decoupled positional K
        # --------------------------------------------------

        self.k_rope_proj = nn.Linear(
            d_model,
            rope_dim,
            bias=False,
        )

        self.rope = RotaryEmbedding(
            rope_dim,
            max_seq_len=max_seq_len,
        )

        # --------------------------------------------------
        # Output
        # --------------------------------------------------

        self.out_proj = nn.Linear(
            d_model,
            d_model,
            bias=False,
        )

    def forward(
        self,
        x,
        past_kv=None,
        use_cache=False,
    ):

        B, T, _ = x.shape

        # ==================================================
        # QUERY
        # ==================================================

        q_content = self.q_content_proj(x)

        q_content = q_content.view(
            B,
            T,
            self.n_heads,
            self.content_dim,
        ).transpose(1, 2)

        q_rope = self.q_rope_proj(x)

        q_rope = q_rope.view(
            B,
            T,
            self.n_heads,
            self.rope_dim,
        ).transpose(1, 2)

        # ==================================================
        # CURRENT TOKEN KV LATENT
        # ==================================================

        latent = self.kv_down_proj(x)

        # [B, T, latent_dim]
        k_content = self.k_content_up_proj(latent)

        k_content = k_content.view(
            B,
            T,
            self.n_heads,
            self.content_dim,
        ).transpose(1, 2)

        v = self.v_up_proj(latent)

        v = v.view(
            B,
            T,
            self.n_heads,
            self.head_dim,
        ).transpose(1, 2)

        # ==================================================
        # ROPE
        # ==================================================

        k_rope = self.k_rope_proj(x)

        k_rope = k_rope.unsqueeze(1)

        # During cached generation, x contains only the
        # newly generated token, so its position is past_len.
        past_len = 0 if past_kv is None else past_kv[0].size(1)

        q_rope, k_rope = self.rope(
            q_rope,
            k_rope,
            offset=past_len,
        )

        # k_rope currently:
        # [B, 1, T, rope_dim]

        k_rope = k_rope.expand(
            -1,
            self.n_heads,
            -1,
            -1,
        )

        # ==================================================
        # APPEND CACHE
        # ==================================================

        if past_kv is not None:

            past_latent, past_k_rope = past_kv

            latent = torch.cat(
                [past_latent, latent],
                dim=1,
            )

            k_rope = torch.cat(
                [past_k_rope, k_rope],
                dim=2,
            )

            # Reconstruct all historical K/V
            k_content = self.k_content_up_proj(latent)

            total_len = latent.size(1)

            k_content = k_content.view(
                B,
                total_len,
                self.n_heads,
                self.content_dim,
            ).transpose(1, 2)

            v = self.v_up_proj(latent)

            v = v.view(
                B,
                total_len,
                self.n_heads,
                self.head_dim,
            ).transpose(1, 2)

        # ==================================================
        # CURRENT QUERY ROPE
        # ==================================================

        # q_rope has only the current token(s)
        q_rope = q_rope

        # ==================================================
        # CONCAT CONTENT + POSITION
        # ==================================================

        q = torch.cat(
            [
                q_content,
                q_rope,
            ],
            dim=-1,
        )

        k = torch.cat(
            [
                k_content,
                k_rope,
            ],
            dim=-1,
        )

        # ==================================================
        # ATTENTION
        # ==================================================

        if past_kv is None:

            # Full training sequence
            attn_out = F.scaled_dot_product_attention(
                q,
                k,
                v,
                is_causal=True,
            )

        else:

            # Cached generation:
            # q contains only the new token,
            # k/v contain all tokens so far.
            attn_out = F.scaled_dot_product_attention(
                q,
                k,
                v,
                is_causal=False,
            )

        # ==================================================
        # OUTPUT
        # ==================================================

        attn_out = (
            attn_out
            .transpose(1, 2)
            .contiguous()
            .view(B, T, self.d_model)
        )

        out = self.out_proj(attn_out)

        if use_cache:

            new_cache = (
                latent.detach(),
                k_rope.detach(),
            )

            return out, new_cache

        return out
class DecoderBlock(nn.Module):

    def __init__(
        self,
        d_model,
        n_heads,
        kv_latent_dim,
        rope_dim,
        d_ff,
        max_seq_len=2048,
    ):
        super().__init__()

        self.norm1 = nn.RMSNorm(d_model)

        self.attention = MLAAttention(
            d_model=d_model,
            n_heads=n_heads,
            kv_latent_dim=kv_latent_dim,
            rope_dim=rope_dim,
            max_seq_len=max_seq_len,
        )

        self.norm2 = nn.RMSNorm(d_model)

        self.ffn = SwiGLU(
            d_model,
            d_ff,
        )

    def forward(
        self,
        x,
        past_kv=None,
        use_cache=False,
    ):

        attn_result = self.attention(
            self.norm1(x),
            past_kv=past_kv,
            use_cache=use_cache,
        )

        if use_cache:
            attn_out, new_kv = attn_result
        else:
            attn_out = attn_result

        x = x + attn_out

        x = x + self.ffn(
            self.norm2(x)
        )

        if use_cache:
            return x, new_kv

        return x
class EfficientDecoderLM(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model=640,
        n_heads=16,
        n_layers=12,
        d_ff=1728,
        kv_latent_dim=160,
        rope_dim=32,
        max_seq_len=2048,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model

        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model,
        )

        self.blocks = nn.ModuleList([
            DecoderBlock(
                d_model=d_model,
                n_heads=n_heads,
                kv_latent_dim=kv_latent_dim,
                rope_dim=rope_dim,
                d_ff=d_ff,
                max_seq_len=max_seq_len,
            )
            for _ in range(n_layers)
        ])

        self.final_norm = nn.RMSNorm(d_model)

        self.lm_head = nn.Linear(
            d_model,
            vocab_size,
            bias=False,
        )

        self.lm_head.weight = self.token_embedding.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):

        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)

        elif isinstance(module, nn.Embedding):
            nn.init.xavier_uniform_(module.weight)

        elif isinstance(module, nn.RMSNorm):
            nn.init.ones_(module.weight)

    def forward(
        self,
        input_ids,
        past_kvs=None,
        use_cache=False,
    ):

        x = self.token_embedding(input_ids)

        new_kvs = []

        for i, block in enumerate(self.blocks):

            past_kv = (
                None
                if past_kvs is None
                else past_kvs[i]
            )

            if use_cache:

                x, new_kv = block(
                    x,
                    past_kv=past_kv,
                    use_cache=True,
                )

                new_kvs.append(new_kv)

            else:

                x = block(x)

        x = self.final_norm(x)

        logits = self.lm_head(x)

        if use_cache:
            return logits, new_kvs

        return logits

In [26]:
d_model = 640
n_heads = 16
n_layers = 12
d_ff = 1728

kv_latent_dim = 160
rope_dim = 32

In [27]:
model = EfficientDecoderLM(
    vocab_size=VOCAB_SIZE,

    d_model=d_model,
    n_heads=n_heads,
    n_layers=n_layers,
    d_ff=d_ff,

    # MLA
    kv_latent_dim=kv_latent_dim,
    rope_dim=rope_dim,

    max_seq_len=SEQ_LEN,
).to(DEVICE)

print(
    f"Parameters: "
    f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M"
)

Parameters: 73.84M


In [28]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=0.1,
)


In [29]:
import time
import torch
import torch.nn.functional as F
from bert_score import score as bert_score


def train(
    model,
    train_loader,
    optimizer,
    val_data,
    train_tokens,
    tag,
    val_interval_tokens=1_000_000,
    resume=False,
    resume_tag=None,
):
    model.train()

    tokens_seen = 0
    global_step = 0
    epoch = 0
    next_val_tokens = val_interval_tokens

    if resume:
        if tag is None:
            raise ValueError("resume_tag must be provided when resume=True")

        state = load_checkpoint(
            tag=resume_tag,
            model=model,
            optimizer=optimizer,
        )

        tokens_seen = state["tokens_seen"]
        global_step = state["global_step"]
        epoch = state["epoch"]

        next_val_tokens = (
            (tokens_seen // val_interval_tokens) + 1
        ) * val_interval_tokens

        print(
            f"Resumed from {resume_tag} | "
            f"Tokens: {tokens_seen:,} | "
            f"Step: {global_step}"
        )

    start_time = time.perf_counter()

    while tokens_seen < train_tokens:

        for input_ids in train_loader:

            if tokens_seen >= train_tokens:
                break

            input_ids = input_ids.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            logits = model(input_ids)

            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                input_ids[:, 1:].reshape(-1),
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP,
            )

            optimizer.step()

            batch_tokens = input_ids.numel()
            tokens_seen += batch_tokens
            global_step += 1

            torch.cuda.synchronize()

            elapsed = time.perf_counter() - start_time
            tokens_per_sec = tokens_seen / elapsed

            print(
                f"Step {global_step:5d} | "
                f"Tokens {tokens_seen:,}/{train_tokens:,} | "
                f"Loss {loss.item():.4f} | "
                f"Time {elapsed:.2f}s | "
                f"Tokens/s {tokens_per_sec:,.0f}"
            )
            
            # inside train(), replace the generation/BERTScore validation block with:
            
            if tokens_seen >= next_val_tokens:
            
                model.eval()
            
                val_loss = 0.0
                val_tokens = 0
            
                with torch.no_grad():
                    for example in val_data:
            
                        prompt_ids = tokenizer.encode(example["prompt"]).ids
                        target_ids = tokenizer.encode(example["target"]).ids
            
                        input_ids = torch.tensor(
                            [prompt_ids + target_ids],
                            dtype=torch.long,
                            device=DEVICE,
                        )
            
                        logits = model(input_ids)
            
                        prompt_len = len(prompt_ids)
            
                        target_logits = logits[:, prompt_len - 1:-1, :]
                        target = input_ids[:, prompt_len:]
            
                        loss = F.cross_entropy(
                            target_logits.reshape(-1, target_logits.size(-1)),
                            target.reshape(-1),
                            reduction="sum",
                        )
            
                        val_loss += loss.item()
                        val_tokens += target.numel()
            
                val_loss /= val_tokens
            
                print(
                    f"Validation @ {tokens_seen:,} tokens | "
                    f"Val Loss: {val_loss:.4f}"
                )
            
                model.train()
            
                save_checkpoint(
                    tag=tag,
                    model=model,
                    optimizer=optimizer,
                    tokens_seen=tokens_seen,
                    global_step=global_step,
                    epoch=epoch,
                )
            
                print(f"Checkpoint saved: {tag}\n")
            
                next_val_tokens += val_interval_tokens

    total_time = time.perf_counter() - start_time

    print("\nTraining complete.")
    print(f"Tokens seen: {tokens_seen:,}")
    print(f"Total time: {total_time:.2f}s")
    print(f"Average tokens/s: {tokens_seen / total_time:,.0f}")

In [30]:
train(
    model=model,
    train_loader=train_loader,
    optimizer=optimizer,
    val_data=val_data,
    train_tokens=100_000_000,
    tag="MLA_100M"
)

Step     1 | Tokens 8,192/100,000,000 | Loss 10.4153 | Time 2.52s | Tokens/s 3,255
Step     2 | Tokens 16,384/100,000,000 | Loss 10.1543 | Time 4.19s | Tokens/s 3,912
Step     3 | Tokens 24,576/100,000,000 | Loss 9.7863 | Time 5.87s | Tokens/s 4,188
Step     4 | Tokens 32,768/100,000,000 | Loss 9.6393 | Time 7.55s | Tokens/s 4,338
Step     5 | Tokens 40,960/100,000,000 | Loss 9.4146 | Time 9.20s | Tokens/s 4,454
Step     6 | Tokens 49,152/100,000,000 | Loss 9.1887 | Time 10.89s | Tokens/s 4,515
Step     7 | Tokens 57,344/100,000,000 | Loss 9.0228 | Time 12.59s | Tokens/s 4,553
Step     8 | Tokens 65,536/100,000,000 | Loss 8.8159 | Time 14.31s | Tokens/s 4,581
Step     9 | Tokens 73,728/100,000,000 | Loss 8.6107 | Time 16.03s | Tokens/s 4,600
Step    10 | Tokens 81,920/100,000,000 | Loss 8.4555 | Time 17.76s | Tokens/s 4,612
Step    11 | Tokens 90,112/100,000,000 | Loss 8.2481 | Time 19.51s | Tokens/s 4,618
Step    12 | Tokens 98,304/100,000,000 | Loss 8.1332 | Time 21.26s | Tokens/s 4,

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploaded: checkpoints/MLA_100M/optimizer.pt


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

... (1,191 outputs left out)

Uploaded: checkpoints/MLA_100M/rng_state.pt


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploaded: checkpoints/MLA_100M/model.pt
Checkpoint saved: MLA_100M


Training complete.
Tokens seen: 100,007,936
Total time: 29605.54s
Average tokens/s: 3,378


In [31]:
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO_ID,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

print("Test file:", test_file_path)

test.json: 0.00B [00:00, ?B/s]

Test file: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Test/test.json


In [32]:
with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Number of test examples:", len(test_data))

Number of test examples: 1000


In [33]:
MAX_GENERATION_TOKENS = 256

In [34]:
@torch.no_grad()
def generate_single(
    prompt,
    model,
    tokenizer,
    device,
    target_words=128,
    temperature=0.8,
    top_p=0.9,
):
    model.eval()

    max_input_len = SEQ_LEN - MAX_GENERATION_TOKENS

    prompt_ids = tokenizer.encode(prompt).ids[:max_input_len]

    input_ids = torch.tensor(
        [prompt_ids],
        dtype=torch.long,
        device=device,
    )

    generated_ids = []

    # ========================================================
    # Process prompt ONCE and build KV cache
    # ========================================================

    logits, past_kvs = model(
        input_ids,
        use_cache=True,
    )

    next_logits = logits[:, -1, :] / temperature

    # ========================================================
    # Autoregressive generation
    # ========================================================

    for _ in range(MAX_GENERATION_TOKENS):

        # Top-p
        sorted_logits, sorted_indices = torch.sort(
            next_logits,
            descending=True,
            dim=-1,
        )

        sorted_probs = torch.softmax(
            sorted_logits,
            dim=-1,
        )

        cumulative_probs = torch.cumsum(
            sorted_probs,
            dim=-1,
        )

        remove = cumulative_probs > top_p
        remove[:, 1:] = remove[:, :-1].clone()
        remove[:, 0] = False

        sorted_logits[remove] = float("-inf")

        probs = torch.softmax(
            sorted_logits,
            dim=-1,
        )

        sampled = torch.multinomial(probs, 1)

        next_token = torch.gather(
            sorted_indices,
            -1,
            sampled,
        )

        generated_ids.append(
            next_token.item()
        )

        # Stop at 128 words
        text = tokenizer.decode(generated_ids)

        if len(text.split()) >= target_words:
            break

        # ----------------------------------------------------
        # ONLY THE NEW TOKEN
        # ----------------------------------------------------

        logits, past_kvs = model(
            next_token,
            past_kvs=past_kvs,
            use_cache=True,
        )

        next_logits = logits[:, -1, :] / temperature

    # ========================================================
    # Final prediction
    # ========================================================

    text = tokenizer.decode(generated_ids)

    text = " ".join(
        text.split()[:target_words]
    )

    return text

In [35]:
len(test_data['1']['text'].split())

512

In [36]:
predictions = {}

for sample_id, sample in test_data.items():

    prediction = generate_single(
        prompt=sample["text"],
        model=model,
        tokenizer=tokenizer,
        device=DEVICE,
        target_words=128,
        temperature=0.8,
        top_p=0.9,
    )

    predictions[sample_id] = {
        "text": prediction
    }

    print(f"Finished {sample_id}")

Finished 1
Finished 2
Finished 3
Finished 4
Finished 5
Finished 6
Finished 7
Finished 8
Finished 9
Finished 10
Finished 11
Finished 12
Finished 13
Finished 14
Finished 15
Finished 16
Finished 17
Finished 18
Finished 19
Finished 20
Finished 21
Finished 22
Finished 23
Finished 24
Finished 25
Finished 26
Finished 27
Finished 28
Finished 29
Finished 30
Finished 31
Finished 32
Finished 33
Finished 34
Finished 35
Finished 36
Finished 37
Finished 38
Finished 39
Finished 40
Finished 41
Finished 42
Finished 43
Finished 44
Finished 45
Finished 46
Finished 47
Finished 48
Finished 49
Finished 50
Finished 51
Finished 52
Finished 53
Finished 54
Finished 55
Finished 56
Finished 57
Finished 58
Finished 59
Finished 60
Finished 61
Finished 62
Finished 63
Finished 64
Finished 65
Finished 66
Finished 67
Finished 68
Finished 69
Finished 70
Finished 71
Finished 72
Finished 73
Finished 74
Finished 75
Finished 76
Finished 77
Finished 78
Finished 79
Finished 80
Finished 81
Finished 82
Finished 83
Finished 84
F

In [37]:
predictions

{'1': {'text': 'not necessary for us to direct the State to decide the issue and the challenge to the said notification in respect of the said relief as well as the objection filed by the petitioner and also the reliefs prayed for. The petitioners are directed to be put to regular and complete the consolidation proceedings within a time bound period." This Court is of the considered opinion that the question as to whether the reliefs claimed by the petitioners, in respect of the relief claimed by the petitioner in respect of the relief claimed by the petitioners in respect of the relief claimed by the petitioners before the State Government was barred by laches. Therefore, we are of the considered opinion that the petitioners are entitled to a relief'},
 '2': {'text': "to conduct the investigation of Case Crime No.1148/13, under Sections 406, 420, 467, 468, 471 and 120B of Indian Penal Code. Shri B.K. Shukla, counsel for the applicants has submitted that the complaint case was filed ag

In [38]:
import zipfile
OUTPUT_DIR = "/kaggle/working/cs781_phase1"
PREDICTION_PATH = os.path.join(
    OUTPUT_DIR,
    "prediction.json"
)
SUBMISSION_ZIP = os.path.join(
    OUTPUT_DIR,
    "submission.zip"
)
os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)
with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(
        predictions,
        f,
        ensure_ascii=False,
        indent=2
    )

print("prediction.json saved:")
print(PREDICTION_PATH)

with zipfile.ZipFile(
    SUBMISSION_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zipf:
    zipf.write(
        PREDICTION_PATH,
        arcname="prediction.json"
    )

print("Codabench submission created:")
print(SUBMISSION_ZIP)

prediction.json saved:
/kaggle/working/cs781_phase1/prediction.json
Codabench submission created:
/kaggle/working/cs781_phase1/submission.zip
